In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb

import gc


In [ ]:

# 1


phys_files = [
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Physical\utf16\phy_att_1.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Physical\utf16\phy_att_2.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Physical\utf16\phy_att_3.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Physical\utf16\phy_norm.csv",
]

phys_list = [
    pd.read_csv(path, sep="\t", encoding="utf-16")
    for path in phys_files
]

phy_raw = pd.concat(phys_list, ignore_index=True)

print("RAW physical shape:", phy_raw.shape)
print("RAW physical columns:")
print(phy_raw.columns.tolist())
display(phy_raw.head())

if "Label" in phy_raw.columns:
    print("\nPhysical Label value counts:")
    print(phy_raw["Label"].value_counts())


RAW physical shape: (9206, 43)
RAW physical columns:
['Time', 'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7', 'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6', 'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13', 'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19', 'Valv_20', 'Valv_21', 'Valv_22', 'Label_n', 'Label']


,Time,Tank_1,Tank_2,Tank_3,Tank_4,Tank_5,Tank_6,Tank_7,Tank_8,Pump_1,...,Valv_15,Valv_16,Valv_17,Valv_18,Valv_19,Valv_20,Valv_21,Valv_22,Label_n,Label
0,09/04/2021 18:23:28,0,0,0,0,0,0,0,0,False,...,False,False,False,False,False,False,False,False,0,normal
1,09/04/2021 18:23:29,0,0,0,0,0,0,0,0,False,...,False,False,False,False,False,False,False,False,0,normal
2,09/04/2021 18:23:30,0,0,0,0,0,0,0,0,False,...,False,False,False,False,False,False,False,False,0,normal
3,09/04/2021 18:23:31,0,0,0,0,0,0,0,0,False,...,False,False,False,False,False,False,False,False,0,normal
4,09/04/2021 18:23:32,0,0,0,0,0,0,0,0,True,...,False,False,False,False,False,False,False,False,0,normal



Physical Label value counts:
Label
normal            7498
MITM               743
physical fault     552
nomal              249
DoS                157
scan                 7
Name: count, dtype: int64


In [ ]:

# 2


net_files = [
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_1.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_2.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_3.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_4.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\normal_1.csv",
    r"C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\normal.csv",
]


use_cols = [
    "Time", "mac_s", "mac_d", "ip_s", "ip_d",
    "sport", "dport", "proto", "size", "label", "label_n",
]

net_list = []
for path in net_files:
    print(f"\nReading: {path}")
    df_tmp = pd.read_csv(
        path,
        encoding="utf-8-sig",    # teammate’s tip
        skipinitialspace=True,   # fixes " mac_s" -> "mac_s"
        usecols=use_cols,
    )
    print("  shape:", df_tmp.shape)
    print("  columns:", df_tmp.columns.tolist())
    display(df_tmp.head())
    net_list.append(df_tmp)

network_raw = pd.concat(net_list, ignore_index=True)

print("\nFULL RAW NETWORK SHAPE:", network_raw.shape)
print("FULL RAW NETWORK COLUMNS:")
print(network_raw.columns.tolist())



Reading: C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_1.csv
  shape: (5527409, 11)
  columns: ['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'size', 'label_n', 'label']


,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,size,label_n,label
0,2021-04-09 18:23:28.385003,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,56667.0,502.0,Modbus,66,0,normal
1,2021-04-09 18:23:28.385005,74:46:a0:bd:a7:1b,e6:3f:ac:c9:a8:8c,84.3.251.20,84.3.251.101,56666.0,502.0,Modbus,66,0,normal
2,2021-04-09 18:23:28.385006,74:46:a0:bd:a7:1b,fa:00:bc:90:d7:fa,84.3.251.20,84.3.251.103,56668.0,502.0,Modbus,66,0,normal
3,2021-04-09 18:23:28.385484,0a:fe:ec:47:74:fb,74:46:a0:bd:a7:1b,84.3.251.102,84.3.251.20,502.0,56667.0,Modbus,64,0,normal
4,2021-04-09 18:23:28.385486,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502.0,56668.0,Modbus,64,0,normal



Reading: C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_2.csv
  shape: (5159469, 11)
  columns: ['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'size', 'label_n', 'label']


,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,size,label_n,label
0,2021-04-19 15:37:19.989214,00:80:f4:03:fb:12,74:46:a0:bd:a7:1b,84.3.251.18,84.3.251.20,502.0,61315.0,Modbus,64,0,normal
1,2021-04-19 15:37:19.990641,74:46:a0:bd:a7:1b,e6:3f:ac:c9:a8:8c,84.3.251.20,84.3.251.101,61316.0,502.0,Modbus,66,0,normal
2,2021-04-19 15:37:19.990645,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,61318.0,502.0,Modbus,66,0,normal
3,2021-04-19 15:37:19.990647,74:46:a0:bd:a7:1b,fa:00:bc:90:d7:fa,84.3.251.20,84.3.251.103,61317.0,502.0,Modbus,66,0,normal
4,2021-04-19 15:37:19.990943,e6:3f:ac:c9:a8:8c,74:46:a0:bd:a7:1b,84.3.251.101,84.3.251.20,502.0,61316.0,Modbus,65,0,normal



Reading: C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\attack_3.csv
  shape: (5862547, 11)
  columns: ['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'size', 'label_n', 'label']


...


Reading: C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\normal_1.csv
  shape: (4656241, 11)
  columns: ['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'size', 'label_n', 'label']


,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,size,label_n,label
0,2021-04-09 11:53:40.851289,e6:3f:ac:c9:a8:8c,74:46:a0:bd:a7:1b,84.3.251.101,84.3.251.20,502,61515,Modbus,64,0,normal
1,2021-04-09 11:53:40.851289,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502,61516,Modbus,65,0,normal
2,2021-04-09 11:53:40.852148,74:46:a0:bd:a7:1b,00:80:f4:03:fb:12,84.3.251.20,84.3.251.18,61514,502,Modbus,66,0,normal
3,2021-04-09 11:53:40.853127,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,61517,502,Modbus,66,0,normal
4,2021-04-09 11:53:40.853447,0a:fe:ec:47:74:fb,74:46:a0:bd:a7:1b,84.3.251.102,84.3.251.20,502,61517,Modbus,65,0,normal



Reading: C:\Users\amira\CISProj\CIS\data\testbed_system_1\Network\csv\normal.csv
  shape: (3101048, 11)
  columns: ['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'size', 'label_n', 'label']


,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,size,label_n,label
0,2021-04-09 11:30:52.716203,74:46:a0:bd:a7:1b,fa:00:bc:90:d7:fa,84.3.251.20,84.3.251.103,61516,502,Modbus,66,0,normal
1,2021-04-09 11:30:52.716499,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502,61516,Modbus,65,0,normal
2,2021-04-09 11:30:52.717334,74:46:a0:bd:a7:1b,e6:3f:ac:c9:a8:8c,84.3.251.20,84.3.251.101,61515,502,Modbus,66,0,normal
3,2021-04-09 11:30:52.717624,e6:3f:ac:c9:a8:8c,74:46:a0:bd:a7:1b,84.3.251.101,84.3.251.20,502,61515,Modbus,65,0,normal
4,2021-04-09 11:30:52.717952,00:80:f4:03:fb:12,74:46:a0:bd:a7:1b,84.3.251.18,84.3.251.20,502,61514,Modbus,64,0,normal



FULL RAW NETWORK SHAPE: (29829204, 11)
FULL RAW NETWORK COLUMNS:
['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'size', 'label_n', 'label']


In [ ]:

# 3



network_raw = network_raw.dropna(subset=["label"])

print("Full network_raw rows:", len(network_raw))
print("Label distribution (full):")
print(network_raw["label"].value_counts())


SAMPLE_FRAC = 0.03

network_sampled = (
    network_raw
    .groupby("label", group_keys=False)
    .apply(lambda df: df.sample(frac=SAMPLE_FRAC, random_state=42))
    .reset_index(drop=True)
)

print("\nSampled network rows:", len(network_sampled))
print("Label distribution (sample):")
print(network_sampled["label"].value_counts())

display(network_sampled.head())


Full network_raw rows: 29829204
Label distribution (full):
label
normal            20453299
DoS                5671542
MITM               2155409
physical fault     1548504
anomaly                389
scan                    61
Name: count, dtype: int64


C:\Users\amira\AppData\Local\Temp\ipykernel_20868\448899329.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(frac=SAMPLE_FRAC, random_state=42))



Sampled network rows: 894876
Label distribution (sample):
label
normal            613599
DoS               170146
MITM               64662
physical fault     46455
anomaly               12
scan                   2
Name: count, dtype: int64


,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,size,label_n,label
0,2021-04-09 19:55:40.859372,00:0c:29:47:8c:22,00:80:f4:03:fb:12,84.3.251.110,84.3.251.18,NaN,NaN,IP,1514,1,DoS
1,2021-04-09 19:55:54.111694,00:0c:29:47:8c:22,00:80:f4:03:fb:12,84.3.251.110,84.3.251.18,NaN,NaN,IP,1514,1,DoS
2,2021-04-19 16:12:05.956286,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,61318.0,502.0,Modbus,66,1,DoS
3,2021-04-09 19:55:59.775548,00:0c:29:47:8c:22,00:80:f4:03:fb:12,84.3.251.110,84.3.251.18,NaN,NaN,IP,1514,1,DoS
4,2021-04-19 16:00:02.011804,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502.0,61317.0,Modbus,65,1,DoS


In [ ]:

# 4


print("\nParsing mixed-format timestamps in network_sampled...")


network_sampled["Time"] = pd.to_datetime(
    network_sampled["Time"].astype(str),
    format="mixed",     
    errors="coerce"      
)


nat_count = network_sampled["Time"].isna().sum()
print("Number of NaT (unparsed Time values):", nat_count)


network_sampled["bucket"] = network_sampled["Time"].dt.floor("S")

print("\nnetwork_sampled with bucket():")
display(network_sampled[["Time", "bucket", "label"]].head())



Parsing mixed-format timestamps in network_sampled...
Number of NaT (unparsed Time values): 0

network_sampled with bucket():


C:\Users\amira\AppData\Local\Temp\ipykernel_20868\3511010362.py:19: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  network_sampled["bucket"] = network_sampled["Time"].dt.floor("S")


,Time,bucket,label
0,2021-04-09 19:55:40.859372,2021-04-09 19:55:40,DoS
1,2021-04-09 19:55:54.111694,2021-04-09 19:55:54,DoS
2,2021-04-19 16:12:05.956286,2021-04-19 16:12:05,DoS
3,2021-04-09 19:55:59.775548,2021-04-09 19:55:59,DoS
4,2021-04-19 16:00:02.011804,2021-04-19 16:00:02,DoS


In [ ]:

# 5


phy_raw["Time"] = pd.to_datetime(phy_raw["Time"], dayfirst=True)
phy_raw["bucket"] = phy_raw["Time"].dt.floor("S")

phys_feature_cols = [
    c for c in phy_raw.columns
    if c not in ["Time", "Label", "Label_n", "bucket"]
]

phy_physagg = (
    phy_raw
    .groupby("bucket")
    .agg(
        {
            **{c: "mean" for c in phys_feature_cols},
            "Label": "last",
        }
    )
    .reset_index()
)


phy_physagg["Label"] = phy_physagg["Label"].replace({"nomal": "normal"})

print("phy_physagg shape:", phy_physagg.shape)
print("phy_physagg columns:", phy_physagg.columns.tolist())
display(phy_physagg.head())


phy_physagg shape: (9206, 42)
phy_physagg columns: ['bucket', 'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7', 'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6', 'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13', 'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19', 'Valv_20', 'Valv_21', 'Valv_22', 'Label']


C:\Users\amira\AppData\Local\Temp\ipykernel_20868\250256295.py:6: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  phy_raw["bucket"] = phy_raw["Time"].dt.floor("S")


,bucket,Tank_1,Tank_2,Tank_3,Tank_4,Tank_5,Tank_6,Tank_7,Tank_8,Pump_1,...,Valv_14,Valv_15,Valv_16,Valv_17,Valv_18,Valv_19,Valv_20,Valv_21,Valv_22,Label
0,2021-04-09 11:30:50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
1,2021-04-09 11:30:51,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
2,2021-04-09 11:30:52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
3,2021-04-09 11:30:53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
4,2021-04-09 11:30:54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal


In [ ]:

# 6


network_sampled["bucket"] = pd.to_datetime(network_sampled["bucket"])
phy_physagg["bucket"]    = pd.to_datetime(phy_physagg["bucket"])

net_sorted  = network_sampled.sort_values("bucket")
phys_sorted = phy_physagg.sort_values("bucket")

combined_raw = pd.merge_asof(
    net_sorted,
    phys_sorted,
    on="bucket",
    direction="backward",  
)

print("combined_raw shape:", combined_raw.shape)
display(combined_raw.head())


combined_raw shape: (894876, 53)


,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,size,label_n,...,Valv_14,Valv_15,Valv_16,Valv_17,Valv_18,Valv_19,Valv_20,Valv_21,Valv_22,Label
0,2021-04-09 11:30:52.799595,0a:fe:ec:47:74:fb,74:46:a0:bd:a7:1b,84.3.251.102,84.3.251.20,502.0,61517.0,Modbus,65,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
1,2021-04-09 11:30:52.729608,e6:3f:ac:c9:a8:8c,74:46:a0:bd:a7:1b,84.3.251.101,84.3.251.20,502.0,61515.0,Modbus,65,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
2,2021-04-09 11:30:52.755634,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502.0,61516.0,Modbus,65,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
3,2021-04-09 11:30:52.903292,74:46:a0:bd:a7:1b,00:80:f4:03:fb:12,84.3.251.20,84.3.251.18,61514.0,502.0,Modbus,66,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
4,2021-04-09 11:30:52.877694,0a:fe:ec:47:74:fb,74:46:a0:bd:a7:1b,84.3.251.102,84.3.251.20,502.0,61517.0,Modbus,65,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal


In [ ]:
# 7


def map_attack_types_raw(s):
    if pd.isna(s):
        return "Normal"
    s = str(s).lower()

    if "scan" in s:
        return "Scan"
    if "dos" in s:
        return "DoS"
    if "mitm" in s:
        return "MITM"
    if "physical" in s:
        return "Physical Fault"
    if "normal" in s or "nomal" in s:
        return "Normal"
    if "anomaly" in s:
        return "Anomaly"

    return s.title()

combined_raw["attack_class"] = combined_raw["label"].apply(map_attack_types_raw)

print("Attack_class distribution (combined_raw):")
print(combined_raw["attack_class"].value_counts())


Attack_class distribution (combined_raw):
attack_class
Normal            613611
DoS               170146
MITM               64662
Physical Fault     46455
Scan                   2
Name: count, dtype: int64


In [ ]:

# 8


valid_classes = ["Normal", "DoS", "MITM", "Physical Fault", "Scan"]
combined_raw_filtered = combined_raw[combined_raw["attack_class"].isin(valid_classes)].copy()

print("Filtered attack_class value counts:")
print(combined_raw_filtered["attack_class"].value_counts())

y = combined_raw_filtered["attack_class"]

drop_cols = [
    "Time", "bucket",
    "label", "label_n",
    "Label",        
    "attack_class",
    "ip_s", "ip_d",
    "mac_s", "mac_d",
]

X = combined_raw_filtered.drop(columns=[c for c in drop_cols if c in combined_raw_filtered.columns])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print("Final X shape (RAW pipeline):", X.shape)
print("Feature columns:")
print(X.columns.tolist())


Filtered attack_class value counts:
attack_class
Normal            613611
DoS               170146
MITM               64662
Physical Fault     46455
Scan                   2
Name: count, dtype: int64
Final X shape (RAW pipeline): (894876, 44)
Feature columns:
['sport', 'dport', 'proto', 'size', 'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7', 'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6', 'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13', 'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19', 'Valv_20', 'Valv_21', 'Valv_22']


In [ ]:

# 9


le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

print("Label classes:", le_y.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42,
)


Label classes: ['DoS' 'MITM' 'Normal' 'Physical Fault' 'Scan']


In [11]:
from sklearn.model_selection import RandomizedSearchCV

base_model = xgb.XGBClassifier(
    objective="multi:softmax",
    eval_metric="mlogloss",
    n_jobs=-1,
)

param_dist = {
    "n_estimators": [300, 500, 700, 900],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
}

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="balanced_accuracy",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Best CV balanced accuracy:", search.best_score_)
print("Best hyperparameters:\n", search.best_params_)

best_model = search.best_estimator_


Fitting 3 folds for each of 20 candidates, totalling 60 fits


C:\Users\amira\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


Best CV balanced accuracy: 0.7621690001174856
Best hyperparameters:
 {'subsample': 0.9, 'n_estimators': 700, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 0.7}


In [ ]:

y_pred_encoded = best_model.predict(X_test)

y_test_labels = le_y.inverse_transform(y_test)
y_pred_labels = le_y.inverse_transform(y_pred_encoded)

bal_acc = balanced_accuracy_score(y_test_labels, y_pred_labels)
print(f"TUNED MODEL - Balanced Accuracy: {bal_acc:.4f}\n")

print("Classification report (tuned model):")
print(classification_report(y_test_labels, y_pred_labels))

labels = le_y.classes_
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

print("\nPer-class TPR / FPR (tuned model):")
for lbl, row in zip(labels, cm_norm):
    tpr = row[list(labels).index(lbl)]
    fpr = 1 - tpr
    print(f"{lbl:15s} TPR={tpr:.4f}  FPR={fpr:.4f}")


TUNED MODEL - Balanced Accuracy: 0.8803

Classification report (tuned model):
                precision    recall  f1-score   support

           DoS       1.00      0.94      0.97     34029
          MITM       1.00      0.78      0.87     12933
        Normal       0.95      1.00      0.97    122723
Physical Fault       1.00      0.80      0.89      9291

      accuracy                           0.96    178976
     macro avg       0.98      0.88      0.93    178976
  weighted avg       0.96      0.96      0.96    178976


Per-class TPR / FPR (tuned model):
DoS             TPR=0.9386  FPR=0.0614
MITM            TPR=0.7792  FPR=0.2208
Normal          TPR=0.9988  FPR=0.0012
Physical Fault  TPR=0.8045  FPR=0.1955
Scan            TPR=nan  FPR=nan


C:\Users\amira\AppData\Local\Temp\ipykernel_20868\3608744372.py:15: RuntimeWarning: invalid value encountered in divide
  cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
